# X (aka Twitter) Post Generator

Iterative Workflow in LangGraph

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

In [ ]:
# Define Model

generator_model = ChatOpenAI(    
    base_url="http://localhost:12434/engines/v1",
    api_key="docker", 
    temperature=0.3, 
    model = "hf.co/bartowski/llama-3.2-1b-instruct-gguf")

evaluator_model = ChatOpenAI(    
    base_url="http://localhost:12434/engines/v1",
    api_key="docker", 
    temperature=0, 
    model = "ai/smollm2:360M-Q4_K_M")

optimizer_model = ChatOpenAI(    
    base_url="http://localhost:12434/engines/v1",
    api_key="docker", 
    temperature=0, 
    model = "ai/smollm2")

In [ ]:
# Define state

from typing import TypedDict, Literal, Annotated
from typing_extensions import NotRequired
import operator

class AgentState(TypedDict):
    topic: str
    tweet: NotRequired[str]
    evaluation: NotRequired[Literal['approved', 'needs_improvement']]
    feedback: NotRequired[str]
    iterations: int
    max_iterations: int

    tweet_history: Annotated[list[int], operator.add]
    feedback_history: Annotated[list[int], operator.add]


In [ ]:
def generate_tweet(state: AgentState):
    topic = state['topic']
    # prompt
    messages = [
        SystemMessage(content="You are a funny and claver Twitter/X influencer."),
        HumanMessage(content=f"""
        Write a short, original, and hilarious tweet on the topic: {topic}.
        Rules:
        - Do NOT use question-answer format.
        - Max 280 characters.
        - Use observational humor, irony, sarcasm, or cultural reference.
        - Think in meme logic, puchlines, or relatable takes.
        - Use simple day to day english.
        """)
    ]

    # call llm
    tweet = generator_model.invoke(messages).content

    # return result
    return {'tweet': tweet, 'tweet_history': [tweet]}

In [ ]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal['approved', 'needs_improvement']
    feedback: str
    score: int

In [ ]:
strctured_evaluator = evaluator_model.with_structured_output(TweetEvaluation)

In [ ]:
def evaluate_tweet(state: AgentState):
    tweet = state['tweet']
    # prompt
    messages = [
        SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweet based on humor, originality, virality, and tweet format."),
        HumanMessage(content=f"""
            Evaluate the following tweet:\n
            ### Tweet: {tweet}\n
            ### Use the criteria below to evaluate the tweet:
            1. Originality - Is this fresh, or have you seen it hundred times before?
            2. Humor - Dit it genuinely make you laugh, smile or chukle?
            3. Punchness - Is it short, sharp, and scroll-stopping?
            4. Virality Potential - Would people retweet or share it?
            5. Format - Is it well-formed tweet (not a setup punchline joke, not a Q&A joke, and under 280 characters)?

            \n### Auto-reject if:
            - It is written in question-answer format (e.g. "Why did..." or "What happens when...")
            - It exceeds 280 characters.
            - It reads like a traditional setup-punchline joke.
            - Don't end with generic, throwaway or deflating lines that weaken the humor (e.g. "Masterpieces of the auntie-uncle universe" or vague summaries).env

            \n### Respond only in structured format:
            - evaluation: "approved" or "needs_improvement"
            - feedback: One paragraph explaining strength and weakness.
            - score: Total score from rubic (0 to 5)
            """)

    ]

    # call llm
    response = strctured_evaluator.invoke(messages)

    # return result
    return {'evaluation': response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}

In [ ]:
def optimize_tweet(state: AgentState):
    messages = [
        SystemMessage(content="You punch tweet for virality and humor based on given feedback."),
        HumanMessage(content=f"""
                     Improve tweet based on the feedback:
                     {state['feedback']}

                     Topic: {state['topic']}
                     Original tweet: {state['tweet']}

                     Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280.

                     """)
    ]

    response = optimizer_model.invoke(messages).content
    iterations = 1 + state['iterations']

    return {
        "tweet": response,
        "iterations": iterations,
        "tweet_history": [response]
    }


In [ ]:
def route_evaluation(state: AgentState) -> Literal['approved', 'needs_improvement']:
    if state.get('evaluation') == 'approved':
        return 'approved'
    if state.get('iterations', 0) >= state.get('max_iterations', 0):
        return 'approved'
    return 'needs_improvement'
    

In [ ]:
# Build graph

graph = StateGraph(AgentState)

# Add node
graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

# Add edges
graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', route_evaluation, {'approved': END, 'needs_improvement': 'optimize'})
graph.add_edge('optimize', 'evaluate')

# Compile
workflow = graph.compile()

In [ ]:
# visualize graph
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

In [ ]:
initial_state = {
    "topic": "Bollywood movies",
    "iterations": 1,
    "max_iterations": 5
}

final_state = workflow.invoke(initial_state)
final_state